# Native Pythia optimizer-state reconstruction

The standard Hugging Face model checkpoint provides weights. CPS seeks the Jacobian of the actual optimizer-state map, so this notebook also reconstructs Adam moments from native GPT-NeoX ZeRO partitions.

## Operational warning

Native checkpoints are storage-heavy and checkpoint availability differs by model size. The notebook downloads only configuration, model-state metadata, and optimizer-state partitions selected by the repository helper. Confirm Colab disk capacity before continuing.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — download and inventory the native checkpoint

Set `CPS_NATIVE_REVISION` to a branch exposed by `EleutherAI/neox-ckpt-pythia-70m`. The download library emits its own file progress bars; the notebook adds a final file and byte inventory.

In [ ]:
import os, pathlib
from cps.pythia.checkpoints import download_native_checkpoint

revision = os.environ.get("CPS_NATIVE_REVISION", "step143000")
target = pathlib.Path(f"/content/native-pythia-70m-{revision}")
print(f"[NATIVE] downloading revision={revision} to {target}", flush=True)
result = download_native_checkpoint("EleutherAI/neox-ckpt-pythia-70m", revision, target)
total_bytes = sum(pathlib.Path(path).stat().st_size for path in result.files)
print(f"[NATIVE] files={len(result.files)}; bytes={total_bytes:,}", flush=True)
for path in result.files:
    p=pathlib.Path(path)
    print(f"  {p.relative_to(target)}  {p.stat().st_size/2**20:.2f} MiB", flush=True)

## Stage 2 — bind the native moments to the probe contract

The optimizer step number is derived from the checkpoint revision. CPS aligns native parameter names to the selected model tensors and reports the number of reconstructed partitions in the final manifest.

In [ ]:
import dataclasses
from cps.notebook import show_config
from cps.pythia.config import load_probe_config

base = load_probe_config("subjects/pythia/configs/pythia_70m_native.yaml")
step_number = max(1, int(revision.removeprefix("step")))
state = dataclasses.replace(base.state, native_checkpoint_dir=str(target), step=step_number)
model = dataclasses.replace(base.model, revision=revision)
config = dataclasses.replace(base, state=state, model=model)
show_config(config)

In [ ]:
from cps.pythia.runner import run_probe

output = run_probe(config)
print(f"[NATIVE] evidence root={output}", flush=True)

In [ ]:
from cps.notebook import display_probe_summary
summary = display_probe_summary(output)

## Evidence interpretation

This packet is stronger than the reconstructed-moment smoke test because the first and second moments come from the released training state. It is still a selected-coordinate, projected, batch-local linearization; it is not the full training Jacobian.

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts()
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)